# Experiment 1 — Binary Fake-Detection Baseline (MuRIL)
**HM-800 project · replication baseline**

This notebook fine-tunes MuRIL on the **full CONSTRAINT 2021 Hindi dataset** for binary fake vs non-fake detection.
This is the replication baseline (paper Table 1) — the performance ceiling our fine-grained work compares against.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**. (~20 min total)

In [ ]:
# Cell 1 — GPU check
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Install libraries (~2 min)
!pip -q install transformers==4.44.2 datasets accelerate scikit-learn

In [ ]:
# Cell 3 — Get the CONSTRAINT 2021 Hindi dataset
!git clone -q https://github.com/mohit19014/Hindi-Hostility-Detection-CONSTRAINT-2021.git constraint
import pandas as pd
train = pd.read_csv('constraint/Dataset/train.csv')
valid = pd.read_csv('constraint/Dataset/valid.csv')
test  = pd.read_csv('constraint/Dataset/test.csv')
for df in (train, valid, test):
    df['label'] = df['Labels Set'].str.contains('fake', na=False).astype(int)
print(f"Train {len(train)} | Val {len(valid)} | Test {len(test)}")
print(f"Fake rate — train: {train.label.mean():.1%}, test: {test.label.mean():.1%}")

In [ ]:
# Cell 4 — Tokenize with MuRIL
from transformers import AutoTokenizer
import datasets as hfds

tok = AutoTokenizer.from_pretrained("google/muril-base-cased")

def to_ds(df):
    d = hfds.Dataset.from_pandas(df[['Post','label']].rename(columns={'Post':'text'}))
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=128), batched=True)

ds_train, ds_val, ds_test = to_ds(train), to_ds(valid), to_ds(test)
print(ds_train)

In [ ]:
# Cell 5 — Fine-tune (~12 min on T4)
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=2)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"f1": f1_score(p.label_ids, preds),
            "precision": precision_score(p.label_ids, preds),
            "recall": recall_score(p.label_ids, preds),
            "accuracy": accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="exp1_out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="no",
    logging_steps=50, report_to="none", seed=42)

trainer = Trainer(model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# Cell 6 — Final evaluation on the held-out test set  (THIS IS TABLE 1)
import json
res = trainer.evaluate(ds_test)
print("\n=== EXPERIMENT 1 — TEST RESULTS (paper Table 1) ===")
for k in ["eval_f1","eval_precision","eval_recall","eval_accuracy"]:
    print(f"  {k.replace('eval_','').capitalize():>10}: {res[k]:.4f}")

with open("exp1_results.json","w") as f: json.dump(res, f, indent=2)
print("\nSaved: exp1_results.json")

In [ ]:
# Cell 7 — Confusion matrix + error peek
import numpy as np
from sklearn.metrics import confusion_matrix
preds = np.argmax(trainer.predict(ds_test).predictions, axis=1)
cm = confusion_matrix(test['label'], preds)
print("Confusion matrix [rows=true, cols=pred] (0=not-fake, 1=fake):")
print(cm)
errs = test[test['label'] != preds]
print(f"\nErrors: {len(errs)} / {len(test)}")
errs[['Post','Labels Set','label']].head(10)

## Done
Download **exp1_results.json** from the Files sidebar. That JSON is Table 1 of the paper.